# 🎙️ AI Companion: XTTS v2 Remote Voice Bridge (Universal)

### 🛠️ Setup Instructions
1. **GPU**: `Runtime` > `Change runtime type` > **T4 GPU**.
2. **Secrets (IMPORTANT)**: 
   - **Colab**: Add `NGROK_TOKEN` to **Secrets** (key icon) and enable access.
   - **Kaggle**: Add `NGROK_TOKEN` to **Add-ons > Secrets** and enable access.
3. **Internet**: (Kaggle Only) Ensure **Internet ON** is toggled in the right sidebar.
4. **Run All**: `Ctrl + F9`.

**Note**: This version uses optimized environment setup for T4 GPUs.

In [ ]:
# @title 1. Prepare Environment
import os

print("Installing dependencies...")
!pip install -q fastapi uvicorn pyngrok python-multipart TTS torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("\n✅ Environment Ready!")

In [ ]:
# @title 2. Create Bridge Script
with open("bridge_server.py", "w") as f:
    f.write("""
import torch
import os
import shutil
import uuid
from typing import List
from TTS.api import TTS
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import uvicorn
import io

os.environ["COQUI_TOS_AGREED"] = "1"

SPEAKERS_DIR = "speakers"
os.makedirs(SPEAKERS_DIR, exist_ok=True)

print("-> Loading XTTS v2 model... (This may take a minute)")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Model Loaded.")

app = FastAPI()

@app.get("/check_speaker/{speaker_id}")
async def check_speaker(speaker_id: str):
    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    if os.path.exists(speaker_path) and os.listdir(speaker_path):
        return {"exists": True, "speaker_id": speaker_id}
    return {"exists": False, "speaker_id": speaker_id}

@app.post("/upload_speaker")
async def upload_speaker(
    speaker_id: str = Form(...),
    files: List[UploadFile] = File(...)
):
    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    os.makedirs(speaker_path, exist_ok=True)

    # Clear existing files for this speaker
    for f in os.listdir(speaker_path):
        try: os.remove(os.path.join(speaker_path, f))
        except: pass

    for upload in files:
        file_path = os.path.join(speaker_path, upload.filename)
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(upload.file, buffer)

    print(f"✅ Speaker '{speaker_id}' registered with {len(files)} samples.")
    return {"status": "success", "speaker_id": speaker_id}

@app.post("/generate_tts")
async def generate_tts_endpoint(
    text: str = Form(...),
    language: str = Form("en"),
    speaker_id: str = Form(None)
):
    if not speaker_id:
        raise HTTPException(status_code=400, detail="speaker_id is required")

    speaker_path = os.path.join(SPEAKERS_DIR, speaker_id)
    if not os.path.exists(speaker_path):
        raise HTTPException(status_code=404, detail=f"Speaker '{speaker_id}' not found.")

    speaker_wavs = [os.path.join(speaker_path, f) for f in os.listdir(speaker_path) if f.endswith(".wav")]

    # Streaming generator
    def stream_audio():
        import numpy as np
        # 1. Pre-calculate conditioning latents from the speaker wavs
        gpt_cond_latent, speaker_embedding = tts_model.synthesizer.tts_model.get_conditioning_latents(audio_path=speaker_wavs)

        # 2. Start the stream using the pre-calculated latents
        chunks = tts_model.synthesizer.tts_model.inference_stream(
            text=text,
            language=language,
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            stream_chunk_size=20
        )
        for chunk in chunks:
            # Convert float32 tensor to int16 for standard WAV compatibility
            chunk = chunk.cpu().numpy()
            chunk = (chunk * 32767).astype(np.int16)
            yield chunk.tobytes()

    return StreamingResponse(stream_audio(), media_type="audio/l16; rate=24000")
""")
print("✅ Bridge script created.")

In [ ]:
# @title 3. Start XTTS Bridge
import os, sys, time
from pyngrok import ngrok

# Detect Environment and Setup Universal Secret Retrieval
try:
    from google.colab import userdata
    ENV_TYPE = "colab"
except ImportError:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        ENV_TYPE = "kaggle"
    except ImportError:
        ENV_TYPE = "local"

def get_secret(name):
    val = os.environ.get(name)
    if val: return val.strip()
    if ENV_TYPE == "colab":
        try: 
            s = userdata.get(name)
            return s.strip() if s else None
        except: pass
    elif ENV_TYPE == "kaggle":
        try: 
            s = user_secrets.get_secret(name)
            return s.strip() if s else None
        except: pass
    return None

NGROK_TOKEN = get_secret('NGROK_TOKEN')
if NGROK_TOKEN:
    print(f"Applying Ngrok Token (len: {len(NGROK_TOKEN)})...")
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("❌ ERROR: NGROK_TOKEN not found!")

ngrok.kill()
public_url = ngrok.connect(8001).public_url

print("="*50)
print(f"\n🚀 XTTS BRIDGE ONLINE!\n")
print(f"URL: {public_url}\n")
print("="*50)

# Run server
!uvicorn bridge_server:app --host 0.0.0.0 --port 8001 --log-level info --timeout-keep-alive 75